# 단지별 세대수 / 도시별 세대수 변수 추가
- 국토교통부와 건축물대장 CSV를 이용해 단지명을 매칭하여 세대수 변수값 수집, 단지별_세대수 값을 가지고 도시별_세대수 값 수집
- `단지별_세대수`, `도시별_세대수` 컬럼 추가
- 단지별_세대수에 쓰여진 함수 설명

    tokenize()
    → 단지명을 토큰으로 쪼개는 함수

    token_similarity()
    → 두 단지명의 유사도 점수 계산
    → tokenize() 사용

    best_match_from()
    → df 전체에서 가장 유사한 단지명 찾기
    → token_similarity() 사용

    find_best_match()  ← 메인 매칭 함수
    → 위 함수들을 모두 조합해서 최종 세대수 반환
    → best_match_from() 사용

- 단지별 세대수

In [56]:
import pandas as pd
import re

In [57]:
# 데이터 불러오기 (원본 데이터의 단지명과 정확히 일치하는 단지명을 찾기 위해서 두개의 CSV 데이터 이용)

# 국토교통부_공동주택 단지 기본 정보 csv 불러오기
kapt_df = pd.read_csv("국토교통부_공동주택_단지기본정보.csv", header=1, skiprows=[2], encoding="cp949")

# 건축물대장 표제부 csv 불러오고 동별 세대수를 합쳐서 단지별 세대수로 만들기
건축물대장_df = pd.read_csv(r"건축물대장_표제부.csv", encoding="utf-8-sig", low_memory=False)
건축물_households = (
    건축물대장_df[건축물대장_df["세대수(세대)"] > 0]
    .groupby("건물명", as_index=False)["세대수(세대)"]
    .sum()
    .rename(columns={"건물명": "단지명", "세대수(세대)": "세대수"})
)

# 변수 추가할 CSV 데이터 불러오기
df = pd.read_csv("new_city2.csv", encoding="utf-8-sig")
print("데이터 크기:", df.shape)
df.head()

C:\Users\2471369\AppData\Local\Temp\ipykernel_35556\1103095700.py:4: DtypeWarning: Columns (0: 연간소독횟수, 1: 전기-수전용량) have mixed types. Specify dtype option on import or set low_memory=False.
  kapt_df = pd.read_csv("국토교통부_공동주택_단지기본정보.csv", header=1, skiprows=[2], encoding="cp949")


데이터 크기: (50307, 17)


,도시명,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),동,층,건축년도,도로명,자족용지비율,단지별_세대수,도시별_세대수
0,송도,인천광역시 연수구 송도동,2026-03-10 0:00,3,10,송도풍림아이원2단지,84.9400,202605,14,58000,-,3,2005,해돋이로120번길 16,0.12012,632.0,84797
1,송도,인천광역시 연수구 송도동,308-1,308,1,더샵송도마리나베이,84.2020,202605,14,58000,-,5,2020,랜드마크로 160,0.12012,3100.0,84797
2,송도,인천광역시 연수구 송도동,2026-05-17 0:00,17,5,송도더샵하버뷰(D13),101.3190,202605,14,93000,-,7,2011,아트센터대로97번길 75,0.12012,553.0,84797
3,송도,인천광역시 연수구 송도동,2026-02-09 0:00,2,9,송도풍림아이원4단지,84.7159,202605,13,52800,-,2,2005,신송로82번길 6,0.12012,504.0,84797
4,송도,인천광역시 연수구 송도동,161-3,161,3,송도캐슬&해모로,139.0185,202605,12,79000,-,4,2013,송도과학로51번길 136,0.12012,1439.0,84797


In [58]:
# 숫자 추출 함수 (다른 단지끼리 매칭되는걸 막기 위함)
def extract_numbers(name):
    return set(re.findall(r'\d+', str(name)))

In [59]:
# 토큰화 하는 함수
def tokenize(name):
    name = re.sub(r'[()（）]', ' ', str(name))
    tokens = re.findall(r'[가-힣]+|\d+|[a-zA-Z]+', name)
    return set(t.lower() for t in tokens)

# 단지명을 토큰으로 쪼개서 공통 토큰 비율로 유사도를 계산하는 함수
def token_similarity(name1, name2):
    t1 = tokenize(name1)
    t2 = tokenize(name2)

    nums1 = set(t for t in t1 if t.isdigit())
    nums2 = set(t for t in t2 if t.isdigit())
    if nums1 and nums2 and nums1 != nums2:
        return 0.0

    intersection = t1 & t2
    union = t1 | t2
    if not union:
        return 0.0
    return len(intersection) / len(union)

In [60]:
# "경기도 성남시 분당구 백현동" → "분당구", "백현동" 로 추출하는 함수들

# 시군구 키 추출 함수
def get_sigungu_key(sigungu_str):
    parts = str(sigungu_str).split()
    result = ""
    for p in parts:
        if p.endswith(("시", "군", "구")):
            result = p
    return result

# 동리 키 추출 함수
def get_dong_key(sigungu_str):
    parts = str(sigungu_str).split()
    for p in parts:
        if p.endswith(("동", "리", "읍", "면")):
            return p
    return ""

In [61]:
# 단지명 괄호 안 숫자 직접 추출 함수
# 예) 해솔마을1단지(647) → 647

def extract_households_from_name(apt_name):
    match = re.search(r'\((\d+)\)$', str(apt_name))
    return int(match.group(1)) if match else None

In [62]:
# Source_df에서 토큰 유사도 매칭 함수
def best_match_from(apt_name, source_df, cutoff=0.4):
    best_name, best_score = None, 0.0
    for kapt_name in source_df["단지명"].astype(str):
        score = token_similarity(apt_name, kapt_name)
        if score > best_score:
            best_score = score
            best_name = kapt_name
    if best_score >= cutoff:
        households = source_df[source_df["단지명"] == best_name]["세대수"].values[0]
        return best_name, households, best_score
    return None, None, 0.0

In [63]:
# 메인 매칭 함수
# 0순위: 세대수 적혀있는 괄호 안 숫자 직접 추출
# 1순위: kapt_df (시군구+동 필터 후 유사도 매칭)
# 2순위: 건축물대장 (유사도 매칭)

def find_best_match(apt_name, sigungu, kapt_df, 건축물_df, cutoff=0.4):
    # 0순위: 단지명 괄호 안 숫자 직접 추출
    direct = extract_households_from_name(apt_name)
    if direct:
        return apt_name, direct, "괄호추출"

    sigungu_key = get_sigungu_key(sigungu)
    dong_key = get_dong_key(sigungu)

    def apply_filter(source_df):
        filtered = source_df[
            source_df["시군구"].astype(str).str.replace(" ", "").str.contains(
                sigungu_key, na=False
            )
        ] if "시군구" in source_df.columns else source_df

        if dong_key and not filtered.empty:
            col = "법정동주소" if "법정동주소" in source_df.columns else "단지명"
            filtered_dong = filtered[filtered[col].astype(str).str.contains(dong_key, na=False)]
            if not filtered_dong.empty:
                filtered = filtered_dong
        return filtered if not filtered.empty else source_df

    # 1순위: kapt_df
    filtered_kapt = apply_filter(kapt_df)
    name1, h1, s1 = best_match_from(apt_name, filtered_kapt, cutoff)
    if name1:
        return name1, h1, "kapt"

    # 2순위: 건축물대장
    name2, h2, s2 = best_match_from(apt_name, 건축물_households, cutoff)
    if name2:
        return name2, h2, "건축물대장"

    return None, None, "실패"

In [64]:
# 매칭 준비
# 중복 제거한 고유 단지명 + 시군구 목록 추출
unique_combos = df[["단지명", "시군구"]].drop_duplicates()
apt_to_households = {}
unmatched = []

In [65]:
# 매칭 실행
for _, row in unique_combos.iterrows():
    apt_name = row["단지명"]
    sigungu  = row["시군구"]
    matched_name, households, source = find_best_match(apt_name, sigungu, kapt_df, 건축물_households)

    if matched_name:
        apt_to_households[apt_name] = households
        print(f"  ✔ [{source}] {apt_name} → {matched_name} ({int(households)}세대)")
    else:
        apt_to_households[apt_name] = None
        unmatched.append(apt_name)
        print(f"  ✘ [실패] {apt_name}")

  ✔ [kapt] 송도풍림아이원2단지 → 더샵센트럴파크2단지 (632세대)
  ✔ [kapt] 더샵송도마리나베이 → 더샵송도마리나베이 (3100세대)
  ✔ [kapt] 송도더샵하버뷰(D13) → 송도더샵하버뷰13단지 (553세대)
  ✔ [kapt] 송도풍림아이원4단지 → 송도풍림아이원4단지 (504세대)
  ✔ [kapt] 송도캐슬&해모로 → 송도캐슬&해모로 (1439세대)
  ✔ [kapt] 송도웰카운티3단지 → 송도웰카운티3단지 (515세대)
  ✔ [kapt] 힐스테이트레이크송도1차 → 송도더샵그린워크1차 (736세대)
  ✔ [kapt] 송도더샵퍼스트파크F14BL → 송도더샵퍼스트파크14단지 (869세대)
  ✔ [kapt] 더샵그린워크2차 → 더샵그린워크2차 (665세대)
  ✔ [kapt] 인천송도힐스테이트4단지 → 송도웰카운티4단지 (465세대)
  ✔ [kapt] 송도SK뷰센트럴 → 송도 SK VIEW 아파트 (2100세대)
  ✔ [kapt] 송도더샵그린워크3차(18-1블럭) → 송도 더샵 그린워크3차 18-1BL (780세대)
  ✔ [kapt] 송도더샵그린워크3차(17-1블럭) → 송도더샵그린워크3차 D17-1BL (358세대)
  ✔ [kapt] 힐스테이트레이크송도2차 → 힐스테이트레이크송도2차 (889세대)
  ✔ [kapt] 송도코오롱더프라우2단지 → 더샵센트럴파크2단지 (632세대)
  ✔ [kapt] 송도풍림아이원3단지 → 송도더샵마스터뷰 3단지 (478세대)
  ✔ [kapt] 더샵송도프라임뷰20BL → 더샵송도프라임뷰20BL (662세대)
  ✔ [kapt] 더샵그린워크1차 → 송도더샵그린워크1차 (736세대)
  ✔ [kapt] 힐스테이트레이크송도3차 → 더샵 송도센트럴파크 3차 (351세대)
  ✔ [kapt] 송도푸르지오하버뷰 → 송도푸르지오하버뷰 (593세대)
  ✔ [kapt] 송도글로벌파크베르디움 → 송도글로벌파크베르디움 (1153세대)
  ✔ [kapt] 송도풍림아이원1단지 → 송도더샵마스터뷰 1단지 (692세

In [66]:
# 단지별_세대수 컬럼 추가 및 저장
df["단지별_세대수"] = df["단지명"].apply(lambda x: apt_to_households[x])

print(f"\n매칭 실패 단지: {unmatched}")
print(f"세대수 조회 실패 건수: {df['단지별_세대수'].isna().sum()}건")
print(df["단지별_세대수"].describe())

# 매칭 실패 행 제거
df = df.dropna(subset=["단지별_세대수"])
print(f"실패 행 제거 후 남은 행 수: {len(df)}행")

df.to_csv("new_city.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료")


매칭 실패 단지: []
세대수 조회 실패 건수: 0건
count    50307.000000
mean       857.211104
std        490.664101
min         24.000000
25%        553.000000
50%        692.000000
75%       1096.000000
max       3100.000000
Name: 단지별_세대수, dtype: float64
실패 행 제거 후 남은 행 수: 50307행

저장 완료


- 도시별 세대수

In [70]:
# 파주 도시별 세대수 더하기
import pandas as pd

  
# 데이터 불러오기
df = pd.read_csv("경기도_파주시_주민등록월별인구통계_20260506.csv", encoding="utf-8-sig")

  
# 최신 기준 (2026-04) 필터링
latest = df[df["통계연월"] == "2026-04"]

  
# 운정신도시 행정동만 선택
# 운정1~6동 (교하동은 구도심이라 제외)
운정_동 = ["운정1동", "운정2동", "운정3동", "운정4동", "운정5동", "운정6동"]
운정 = latest[latest["읍면동명"].isin(운정_동)]

print(운정[["읍면동명", "세대수", "총인구수"]])
print(f"\n운정신도시 총 세대수: {운정['세대수'].sum():,}세대")

     읍면동명    세대수   총인구수
491  운정1동  24100  58226
492  운정2동  24625  63984
493  운정3동  26933  67097
494  운정4동  10678  24207
495  운정5동  22091  57670
496  운정6동  16178  37832

운정신도시 총 세대수: 124,605세대


In [73]:
import pandas as pd

# 수원시 데이터 불러오기 (광교1동, 광교2동)
df_su = pd.read_csv("경기도_수원시_인구현황_20210531.csv", encoding="cp949")

광교_수원 = df_su[df_su["구_동"].astype(str).str.contains("광교", na=False)]
su_total = 광교_수원["세대수"].sum()

print("수원시 광교 세대수:")
print(광교_수원[["구", "구_동", "세대수"]])
print(f"소계: {su_total:,}세대")

   
# 용인시 데이터 불러오기 (상현1~3동)
# 최신 연도(2024) 기준
df_yo = pd.read_csv("경기도_용인시_인구및세대_현황_20241231.csv", encoding="cp949")

광교_용인 = df_yo[
    (df_yo["기준시점"] == 2024) &
    (df_yo["읍면동"].astype(str).str.contains("상현", na=False))
]
yo_total = 광교_용인["세대수"].sum()

print("\n용인시 상현동 세대수 (2024):")
print(광교_용인[["시군구", "읍면동", "세대수"]])
print(f"소계: {yo_total:,}세대")

   
# 광교신도시 총 세대수
print(f"\n광교신도시 총 세대수: {su_total + yo_total:,}세대")

수원시 광교 세대수:
      구   구_동    세대수
42  영통구  광교1동  21872
43  영통구  광교2동  11598
소계: 33,470세대


FileNotFoundError: [Errno 2] No such file or directory: '경기도_용인시_인구및세대_현황_20241231.csv'

In [ ]:
# 세대수에 대한 데이터를 합하여 딕셔너리 데이터로 만들어 칼럼 추가
# 출처: 공공데이터포털 세대현황 CSV

city_households_map = {
    "판교":  62588,   # 
    "광교":  31000,   # 경기도 공식
    "청라":  44360,   # 인천광역시 공식
    "운정":  124605,   # 파주시 공식
    "송도":  84797,   # 인천경제자유구역청 공식
    "검단":  53760,
}

df["도시별_세대수"] = df["도시명"].map(city_households_map)

print(df[["도시명", "도시별_세대수"]].drop_duplicates())
df.to_csv("new_city.csv", index=False, encoding="utf-8-sig")
print("저장 완료!")

      도시명  도시별_세대수
0      송도    84797
15861  운정   124605
31346  광교    31000
35037  판교    62588
40868  청라    44360
저장 완료!
